# Record the experimental environment

Run this in **the same Kaggle session type** that ran the experiments (GPU on, Internet on), ideally right after one of the experiment notebooks so that the model and fastText caches are present.
It writes `environment.json` and `environment_pip_freeze.txt` to the output folder: Python, PyTorch/Transformers/scikit-learn/mergekit/WECHSEL/FOCUS versions, CUDA and GPU, the Hugging Face checkpoint revisions in the local cache (the ones the run loads) and the current hub heads, dataset revisions, and the SHA-256 of the fastText model used by WECHSEL.

Caveat: the experiment notebooks install packages without version pins, so this records the versions installed **now**. State that in the paper if they differ from the versions of the earlier runs (see `REPRODUCE.md`, section "Checking that a rerun matches").


In [ ]:
%pip install -q transformers datasets scikit-learn
%pip install -q fasttext-numpy2-wheel gensim nltk
%pip install -q wechsel --no-deps
%pip install -q mergekit deepfocus


In [ ]:
# ---- Environment record: versions, hardware, checkpoints, data revisions. Writes environment.json and environment_pip_freeze.txt ----
import sys, os, platform, json, subprocess, hashlib, time
from pathlib import Path

def capture_environment(out_dir, model_ids, fasttext_search_roots=()):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    env = dict(captured_utc=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), python=sys.version.replace("\n", " "),
               platform=platform.platform(), cpu_count=os.cpu_count())
    def pipv(name):
        try:
            from importlib.metadata import version
            return version(name)
        except Exception:
            return None
    names = ["torch", "transformers", "tokenizers", "safetensors", "huggingface_hub", "datasets", "numpy", "scipy", "scikit-learn", "matplotlib",
             "mergekit", "deepfocus", "wechsel", "fasttext", "fasttext-numpy2-wheel", "gensim", "nltk", "accelerate"]
    env["packages"] = {n: pipv(n) for n in names}
    try:
        import torch
        env["torch"] = dict(version=torch.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
                            cuda_available=torch.cuda.is_available(), tf32_matmul=torch.backends.cuda.matmul.allow_tf32,
                            tf32_cudnn=torch.backends.cudnn.allow_tf32, cudnn_deterministic=torch.backends.cudnn.deterministic,
                            deterministic_algorithms=torch.are_deterministic_algorithms_enabled(), num_threads=torch.get_num_threads(),
                            gpus=[dict(name=torch.cuda.get_device_name(i), memory_gb=round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1),
                                       capability=list(torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())])
    except Exception as e:
        env["torch"] = f"unavailable: {type(e).__name__}"
    try:
        env["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                                           capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        env["nvidia_smi"] = f"unavailable: {type(e).__name__}"
    try:
        fr = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=120).stdout
        (out_dir / "environment_pip_freeze.txt").write_text(fr); env["pip_freeze_lines"] = len(fr.splitlines())
    except Exception as e:
        env["pip_freeze_lines"] = f"unavailable: {type(e).__name__}"
    # checkpoints: the revision cached on this machine (the one the run loads) and the current hub head
    env["models"] = {}
    try:
        from huggingface_hub import scan_cache_dir, HfApi
        cached = {}
        try:
            for repo in scan_cache_dir().repos:
                if repo.repo_type == "model": cached[repo.repo_id] = sorted(r.commit_hash for r in repo.revisions)
        except Exception:
            pass
        api = HfApi()
        for m in model_ids:
            d = dict(cached_revisions=cached.get(m))
            try: d["hub_head_now"] = api.model_info(m).sha
            except Exception as e: d["hub_head_now"] = f"unavailable: {type(e).__name__}"
            env["models"][m] = d
        env["datasets"] = {}
        for ds in ("qiaojin/PubMedQA", "Salesforce/wikitext"):
            try: env["datasets"][ds] = api.dataset_info(ds).sha
            except Exception as e: env["datasets"][ds] = f"unavailable: {type(e).__name__}"
    except Exception as e:
        env["models"] = f"unavailable: {type(e).__name__}"
    try:
        import urllib.request
        req = urllib.request.Request("https://api.github.com/repos/Franck-Dernoncourt/pubmed-rct/commits/master", headers={"User-Agent": "env-capture"})
        env["datasets"]["Franck-Dernoncourt/pubmed-rct (master head now)"] = json.load(urllib.request.urlopen(req, timeout=30))["sha"]
    except Exception as e:
        env.setdefault("datasets", {})["Franck-Dernoncourt/pubmed-rct (master head now)"] = f"unavailable: {type(e).__name__}"
    # fastText model used by WECHSEL (cc.en.300.bin): size and SHA-256 if present
    env["fasttext_cc_en_300"] = None
    for root in fasttext_search_roots:
        for p in Path(root).rglob("cc.en.300.bin"):
            h = hashlib.sha256()
            with open(p, "rb") as fh:
                for blk in iter(lambda: fh.read(1 << 24), b""): h.update(blk)
            env["fasttext_cc_en_300"] = dict(path=str(p), bytes=p.stat().st_size, sha256=h.hexdigest()); break
        if env["fasttext_cc_en_300"]: break
    (out_dir / "environment.json").write_text(json.dumps(env, indent=2))
    return env

MODEL_IDS = ["openai-community/gpt2", "openai-community/gpt2-medium", "openai-community/gpt2-large", "facebook/opt-125m",
             "EleutherAI/pythia-160m", "EleutherAI/pythia-410m", "EleutherAI/pythia-1b", "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B"]


In [ ]:
import os
CACHE_ROOT = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
OUT_DIR = "environment_record"
env = capture_environment(OUT_DIR, MODEL_IDS, [CACHE_ROOT, os.path.expanduser("~/.cache"), "/kaggle/temp", "/tmp"])
print(json.dumps({k: env[k] for k in ("python", "platform", "packages", "torch", "nvidia_smi", "datasets", "fasttext_cc_en_300")}, indent=1))
print()
for m, d in env["models"].items(): print(m, d)
